In [4]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers_sae import _autoreload
from transformers_sae.replacement_model import make_replacement_model

# Tweak TRAINING_BATCH_SIZE for your hardware if necessary
if torch.cuda.is_available():
    TRAINING_DEVICE = "cuda:0"
    TRAINING_BATCH_SIZE = 8
elif torch.mps.is_available():
    TRAINING_DEVICE = "mps:0"
    TRAINING_BATCH_SIZE = 4
else:
    TRAINING_DEVICE = "cpu"
    TRAINING_BATCH_SIZE = 4

tokenizer = AutoTokenizer.from_pretrained("gpt2")
training_dataset = load_dataset(
    "monology/pile-uncopyrighted-parquet",
    split="train",
    streaming=True,
    columns=["text"],
)
validation_dataset = load_dataset(
    "monology/pile-test-val",
    split="validation",
    revision="refs/convert/parquet",
    streaming=True,
    columns=["text"],
)
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").to(
    device=TRAINING_DEVICE
)
model = make_replacement_model(
    model,
    {},
    num_layers=model.config.n_layer,
    context_length=model.config.n_ctx,
    d_model=model.config.n_embd,
)

print(model)

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

ReplacementModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [5]:
# TRAINING_CACHE_DIR = None if torch.cuda.is_available() or IN_COLLAB else ".training_cache"
# VALIDATION_CACHE_DIR = None if torch.cuda.is_available() or IN_COLLAB else ".validation_cache"
NUM_TRAINING_TOKENS = int(1e7) if torch.cuda.is_available() else int(1e6)
EVAL_INTERVAL = int(1e5)
NUM_VALIDATION_TOKENS = int(1e6) if torch.cuda.is_available() else int(1e5)
D_SAE = model.d_model * 4
TOPK = 100
TOKENIZER_BATCH_SIZE = 128
FINETUNE_FRACTION = 0.1
# Note this will use up ~1.8GB of space, set to False if you want to skip
SAVE_FINAL_RESULTS = True

In [ ]:
import numpy as np

from transformers_sae.sae import (
    SAE,
    DecoderConfig,
    EncoderConfig,
    SAEConfig,
    TopKActivationFunctionConfig,
)
from transformers_sae.training import TrainingConfig, TrainingMethod, fine_tune, train
from transformers_sae.validation import run_validations

saes = {
    layer: SAE(
        SAEConfig(
            d_model=model.d_model,
            d_sae=D_SAE,
            device=TRAINING_DEVICE,
            encoder=EncoderConfig(
                d_model=model.d_model,
                d_sae=D_SAE,
                device=TRAINING_DEVICE,
                activation_function=TopKActivationFunctionConfig(k=TOPK),
            ),
            decoder=DecoderConfig(
                d_model=model.d_model,
                d_sae=D_SAE,
                device=TRAINING_DEVICE,
            ),
        )
    )
    for layer in range(model.num_layers)
}


def linear_decay_during_finetune(frac_trained: float):
    if frac_trained < (1 - FINETUNE_FRACTION):
        return 1.0
    return 1.0 - (frac_trained - (1 - FINETUNE_FRACTION)) / FINETUNE_FRACTION


training_config = TrainingConfig(
    tokenizer_batch_size=TOKENIZER_BATCH_SIZE,
    training_batch_size=TRAINING_BATCH_SIZE,
    num_train_tokens=NUM_TRAINING_TOKENS,
    eval_interval=EVAL_INTERVAL,
    train_layers=list(range(model.num_layers)),
    lr=1e-3,
    interaction_lr=1e-3,
    lr_schedule=linear_decay_during_finetune,  # per Karvonen (2025)
    downstream_reconstruction_weight=1.0,
    reconstruction_weight=1.0,
    balance_reconstruction_losses=True,
    method=TrainingMethod.standard,
    finetune_fraction=None,
)

In [4]:
for s in saes.values():
    s.init_weights()

Traceback (most recent call last):
  File "/Users/evanlloyd/mechinterp-experiments/transformers_sae/.venv/lib/python3.13/site-packages/jurigged/codetools.py", line 533, in _process_child_correspondence
    orig.apply_correspondence(
    ~~~~~~~~~~~~~~~~~~~~~~~~~^
        ccorr,
        ^^^^^^
        order=order,
        ^^^^^^^^^^^^
        controller=controller,
        ^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/evanlloyd/mechinterp-experiments/transformers_sae/.venv/lib/python3.13/site-packages/jurigged/codetools.py", line 744, in apply_correspondence
    new_obj = self.reevaluate(corr.new.node, glb)
  File "/Users/evanlloyd/mechinterp-experiments/transformers_sae/.venv/lib/python3.13/site-packages/jurigged/codetools.py", line 832, in reevaluate
    exec(code, glb, lcl)
    ~~~~^^^^^^^^^^^^^^^^
  File "/Users/evanlloyd/mechinterp-experiments/transformers_sae/transformers_sae/tokenization.py", line 40, in <adjust>
    max_tokens: Optional[int, float],
                ~~~~~~~~^